# Day 23: Designing Distributed Systems

**Week 4 — System Design**

---

## 📚 Theory: Consistent Hashing & Data Replication

When a single database server can no longer hold all your data, you must **shard** (partition) the data across multiple servers. But how do you decide which server holds which data?

### Consistent Hashing
If we use standard hashing `server_index = hash(key) % N` (where N is the number of servers), adding or removing a single server changes `N`, meaning almost *every* key must be remapped! 
**Consistent Hashing** maps both the keys and the servers to an abstract "hash ring". 
1. Hash the servers to place them on the ring.
2. Hash the key to find its position on the ring.
3. Move clockwise on the ring to find the first server. That server stores the key.
When a server is added or removed, only the keys immediately adjacent to it on the ring are affected. (To prevent uneven data distribution, we use "virtual nodes").

### Quorum Consensus
To ensure High Availability, data is replicated across multiple servers. How do we ensure consistency when multiple clients are reading and writing?
- `N` = Number of replicas
- `W` = A write must be acknowledged by `W` replicas to be considered successful.
- `R` = A read must wait for responses from `R` replicas to be considered successful.

**The Golden Rule**: If `W + R > N`, you are guaranteed strong consistency because there is always an overlapping node that has the latest data!

## 🔨 Exercise 1: Design a Rate Limiter

A Rate Limiter restricts the number of events a user can perform within a specific time window. It prevents DoS attacks and resource starvation.

### 1. Requirements
- Limit requests to 10 requests per second per IP address.
- Must be highly available and distributed across multiple API gateway servers.
- Extremely low latency (cannot slow down valid requests).

### 2. The Token Bucket Algorithm
The most common algorithm used by Amazon and Google.
- Imagine a bucket that holds a maximum of `C` tokens.
- Tokens are refilled at a constant rate of `R` tokens per second.
- When a request arrives, if the bucket has enough tokens, we take 1 token and process the request.
- If the bucket is empty, the request is dropped (HTTP 429 Too Many Requests).

### 3. High-Level Architecture

```mermaid
graph TD
    Client(Client) --> LB[Load Balancer]
    LB --> API1[API Gateway 1]
    LB --> API2[API Gateway 2]
    API1 --> Redis[(Redis)]
    API2 --> Redis
    API1 --> Internal[Internal Services]
```

**Where do we store the rules and counters?**
We cannot store counters in the memory of a single API Gateway because the user's next request might hit a *different* API gateway! 
We must use a centralized fast in-memory cache like **Redis**. Redis is perfect because it supports atomic operations (like `INCR` and `EXPIRE`) and is blazing fast.

**Race Conditions!**
If two API gateways read the Redis counter at the exact same millisecond, they might both see `count=9` and both allow the request, bumping the total to 11. 
*Solution*: Use **Lua Scripts** in Redis to execute the check-and-increment atomically, or use Redis's `MULTI/EXEC` transactions.

## 🔨 Exercise 2: Design a Distributed Key-Value Store

Design something similar to Amazon DynamoDB or Cassandra.

### 1. Requirements
- `get(key)`: Returns value.
- `put(key, value)`: Stores value.
- Highly available, highly scalable, tunable consistency.

### 2. Architecture Decisions
- **Data Partitioning**: Use **Consistent Hashing** with virtual nodes to evenly distribute data across a cluster of servers.
- **Data Replication**: Replicate data across the `N` adjacent nodes on the hash ring. (e.g., if a key hashes to Node A, also store a copy on Node B and Node C).
- **Consistency**: Use **Quorum Consensus** (`W + R > N`). Allow clients to choose: do they want fast reads (`R=1`) or strongly consistent reads (`R=majority`)?
- **Inconsistency Resolution**: In a highly available system, network partitions will happen. Two clients might update the same key on two disconnected servers simultaneously. Use **Vector Clocks** `[Server_A: version 1, Server_B: version 2]` to detect conflicts, and force the client to resolve them on their next read.
- **Failure Detection**: Servers send heartbeats via a **Gossip Protocol** to each other. If a server stops responding, the cluster marks it as dead and routes traffic to replicas.

## 📝 Day 23 Review

### Concepts Validated Today
- [ ] The **Token Bucket** and **Leaky Bucket** rate limiting algorithms.
- [ ] Handling Distributed Race Conditions using Redis and Lua Scripts.
- [ ] Understanding **Consistent Hashing** to minimize data movement when scaling clusters.
- [ ] Using **Quorum Consensus** (`W + R > N`) to guarantee read consistency across replicated databases.
- [ ] Resolving data conflicts using **Vector Clocks**.

### Tomorrow's Preview
**Day 24: Designing an Agent Orchestrator** — Shifting focus specifically to the Google Agent Development role! We will architect a system capable of managing long-running AI agents, handling state, and dealing with asynchronous tool execution.